# metaprivBIDS command-line tutorial

This notebook exercises the current `metaprivBIDS` CLI from end to end.
It uses subprocess commands instead of the Python API, so every operation
shown here can be copied into a terminal. Data and results remain local;
generated files go into a temporary directory and never overwrite the
source dataset.

## Environment

Create the supported Python/R environment before starting Jupyter. These
activation-free commands work in PowerShell, Git Bash, and Linux Bash:

```console
conda env create --file environment.yml
conda run --name metaprivbids uv pip install -e ".[notebook]"
conda run --name metaprivbids jupyter lab MetaprivBIDS_CoreLogic_Tutorial.ipynb
```

SUDA2 uses the Conda-provided R runtime and `sdcMicro`; the other commands
run entirely in Python.

In [ ]:
from pathlib import Path
import shlex
import subprocess
import sys
import tempfile

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Start Jupyter from the metaprivBIDS repository root.")

DATA = ROOT / "Use_Case_Data" / "adult_mini.csv"
OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="metaprivbids_cli_"))

def run_cli(*arguments):
    # Run the CLI with this notebook kernel's Python environment.
    print("$ metaprivBIDS " + " ".join(shlex.quote(str(arg)) for arg in arguments))
    result = subprocess.run(
        [sys.executable, "-m", "metaprivBIDS.cli", *map(str, arguments)],
        cwd=ROOT,
        text=True,
        capture_output=True,
        check=True,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if result.stderr:
        print(result.stderr.rstrip())
    return result

print(f"Input:  {DATA}")
print(f"Output: {OUTPUT_DIR}")

## Discover the available commands

In [ ]:
run_cli("--help")

## Inspect the data

`inspect` reports storage type, missingness, distinct-value count, and the
inferred categorical/continuous role for each column.

In [ ]:
run_cli("inspect", DATA)

## Privacy summary

Quasi-identifiers are passed as a comma-separated list. Supplying a
sensitive attribute adds l-diversity to the output.

In [ ]:
QIS = "age,education,marital-status,occupation,sex"
SENSITIVE = "disease"
run_cli("privacy", DATA, "--columns", QIS, "--sensitive", SENSITIVE)

## Variable contribution

K-global measures the effect of removing each quasi-identifier. K-combined
evaluates variable combinations. Both result tables are exported.

In [ ]:
K_GLOBAL = OUTPUT_DIR / "k_global.csv"
K_COMBINED = OUTPUT_DIR / "k_combined.csv"
run_cli("k-global", DATA, "--columns", QIS, "--output", K_GLOBAL)
run_cli("k-combined", DATA, "--columns", QIS,
        "--min-size", 2, "--max-size", 3, "--output", K_COMBINED)
pd.read_csv(K_GLOBAL)

In [ ]:
pd.read_csv(K_COMBINED).head(10)

## Apply anonymisation transformations

Transformation commands always write a new file. This example chains
categorical generalisation, rounding, and reproducible Gaussian noise.
`remove-decimals` and Laplacian noise are also available.

In [ ]:
GENERALIZED = OUTPUT_DIR / "01_generalized.csv"
ROUNDED = OUTPUT_DIR / "02_rounded.csv"
ANONYMISED = OUTPUT_DIR / "03_anonymised.csv"

run_cli("combine", DATA, "--column", "education",
        "--values", "11th,9th,7th-8th,5th-6th,10th,1st-4th",
        "--replacement", "K-12 education", "--output", GENERALIZED)
run_cli("round", GENERALIZED, "--column", "age", "--exponent", 1,
        "--mode", "nearest", "--output", ROUNDED)
run_cli("noise", ROUNDED, "--column", "age", "--distribution", "gaussian",
        "--scale", 1.5, "--seed", 42, "--output", ANONYMISED)
pd.read_csv(ANONYMISED).head()

## Recalculate privacy after transformation

In [ ]:
run_cli("privacy", ANONYMISED, "--columns", QIS, "--sensitive", SENSITIVE)

## PIF, CIG, RIG, and robust outliers

`cig` prints the requested Personal Information Factor and can export the
full information-gain table, variable summary, and two-sided MAD outliers.

In [ ]:
CIG = OUTPUT_DIR / "cig_rig.csv"
CIG_SUMMARY = OUTPUT_DIR / "cig_summary.csv"
RIG_OUTLIERS = OUTPUT_DIR / "rig_outliers.csv"
run_cli("cig", ANONYMISED, "--columns", QIS, "--percentile", 95,
        "--output", CIG, "--summary-output", CIG_SUMMARY,
        "--outliers-output", RIG_OUTLIERS)

In [ ]:
print("Highest-risk rows:")
display(pd.read_csv(CIG).head())
print("Per-variable CIG summary:")
display(pd.read_csv(CIG_SUMMARY))

## SUDA2 through R/sdcMicro

SUDA2 is the only workflow crossing the Python/R boundary. The CLI exports
row scores, cell percentages, variable contributions, and attribute levels.

In [ ]:
SUDA_SCORES = OUTPUT_DIR / "suda_scores.csv"
SUDA_CELLS = OUTPUT_DIR / "suda_cell_contribution.csv"
SUDA_VARIABLES = OUTPUT_DIR / "suda_variable_contribution.csv"
SUDA_LEVELS = OUTPUT_DIR / "suda_attribute_levels.csv"
run_cli("suda", ANONYMISED, "--columns", QIS, "--sample-fraction", 0.2,
        "--output", SUDA_SCORES,
        "--contribution-percent-output", SUDA_CELLS,
        "--attribute-contributions-output", SUDA_VARIABLES,
        "--attribute-level-output", SUDA_LEVELS)

In [ ]:
print("Highest SUDA2 row scores:")
display(pd.read_csv(SUDA_SCORES).sort_values("dis-score", ascending=False).head())
print("Variable contributions:")
display(pd.read_csv(SUDA_VARIABLES))

## Next steps

Use `metaprivBIDS COMMAND --help` to explore parameters, then translate the
selected sequence into a shell script. Interpret risk metrics in context:
compare privacy improvement with analytical utility lost after each change
instead of treating one metric as a universal pass/fail threshold.

In [ ]:
print(f"Tutorial outputs: {OUTPUT_DIR}")
sorted(path.name for path in OUTPUT_DIR.iterdir())